In [23]:
import pandas as pd

In [24]:
import pandas as pd

countryC = pd.read_csv(
    "https://raw.githubusercontent.com/Imagin-io/country-nationality-list/master/countries.csv"
)

countryC = countryC.rename(
    columns={"alpha_3_code": "country_code", "nationality": "nationality"}
)

countryC = countryC[["country_code", "nationality"]].drop_duplicates()

# Add missing Formula 1 nationality codes
extra_countries = pd.DataFrame({
    "country_code": [
        "DEN", "MON", "NED", "GER", "RSA", "SUI",
        "GBR", "POR", "URS", "ROC", "BUL"
    ],
    "nationality": [
        "Danish", "Monegasque", "Dutch", "German",
        "South African", "Swiss", "British", "Portuguese",
        "Soviet", "Russian", "Bulgarian"
    ]
})

# Add only codes not already present
countryC = pd.concat(
    [
        countryC,
        extra_countries[
            ~extra_countries["country_code"].isin(countryC["country_code"])
        ]
    ],
    ignore_index=True
)

# Remove missing values and duplicates
countryC = countryC.dropna(
    subset=["country_code", "nationality"]
).drop_duplicates(subset=["country_code"])

with open("cleaned/Country_Codes.csv", "w") as f:
    countryC.to_csv(f, index=False)

In [25]:
def endid(df):
    ids = [df.columns[x] for x in range(len(df.columns)) if df.columns[x].endswith("_id")]
    ids2 = [df.columns[x] for x in range(len(df.columns)) if df.columns[x] not in ids]

    return df[ids2 + ids]

In [26]:
import pandas as pd

drivers = pd.read_csv("formula_one_driver.csv").drop(
    columns=["abbreviation", "api_id", "permanent_car_number", "reference", "wikipedia"]
).rename(columns={"id": "driver_id"})

df = pd.read_csv("formula_one_teamdriver.csv").drop(columns=["api_id", "role"])
df = df.drop(columns=["id"])

teams = pd.read_csv("formula_one_team.csv").drop(
    columns=["base_team_id", "api_id", "primary_color", "reference", "wikipedia"]
).rename(columns={"id": "team_id"})

season = pd.read_csv("formula_one_season.csv").drop(
    columns=["api_id", "wikipedia"]
).rename(columns={"id": "season_id"})


drivers = drivers.rename(columns={
    col: f"drivers_{col}" for col in drivers.columns
       if "id" not in col #non contiene "id"
})

cols = [
    "drivers_forename",
    "drivers_surname",
    "drivers_date_of_birth",
    "drivers_nationality",
    "drivers_country_code",
    "driver_id",
]

# Option 1: Reassign directly (recommended)
drivers = drivers[cols]

# fill drivers with missing data
# birthdate
missingBds = {
  "864": "1999-05-06",
  "865": "2001-09-22",
  "866": "2000-10-14",
  "867": "2002-01-31",
  "868": "2004-02-04",
  "869": "2005-05-02",
  "870": "2005-03-07",
  "873": "2004-01-19",
  "874": "2005-11-11",
  "875": "2001-06-16",
  "876": "2002-01-13",
  "877": "1996-05-20",
  "878": "2000-05-23",
  "879": "2000-03-30",
  "880": "2004-12-03",
  "881": "2008-08-09",
  "882": "2006-12-21",
  "883": "2005-03-20"
}

# fill the missing birthdates in the drivers dataframe
for idx, birthdate in missingBds.items():
    drivers.loc[int(idx), "drivers_date_of_birth"] = birthdate

def find_col(df, substring):
    matches = [c for c in df.columns if substring in c]
    if not matches:
        raise KeyError(f"Nessuna colonna contenente '{substring}' in {list(df.columns)}")
    return matches[0]


drv_code_col = find_col(drivers, "country_code")
drv_nat_col = find_col(drivers, "nationality")
cnt_code_col = find_col(countryC, "country_code")
cnt_nat_col = find_col(countryC, "nationality")

# Mappa {country_code: nationality} da countryC (senza duplicati)
nat_map = (
    countryC[[cnt_code_col, cnt_nat_col]]
    .dropna(subset=[cnt_code_col, cnt_nat_col])
    .drop_duplicates(subset=[cnt_code_col])
    .set_index(cnt_code_col)[cnt_nat_col]
    .to_dict()
)

# Aggiorna solo i driver a cui a manca la nazionalità ma hanno un country_code valido
target_ids = drivers.loc[drivers[drv_nat_col].isna() & drivers[drv_code_col].notna(), "driver_id"]
mask = drivers["driver_id"].isin(target_ids)

# Riallinea i country_code mancanti se necessario e poi mappa la nazionalità
drivers.loc[mask, drv_nat_col] = (
    drivers.loc[mask, drv_code_col]
    .map(nat_map)
    .fillna(drivers.loc[mask, drv_nat_col])
)


teams = teams.rename(columns={
    col: f"teams_{col}" for col in teams.columns
        if "id" not in col #non contiene "id"
})

season = season.rename(columns={
    col: f"season_{col}" for col in season.columns
       if "id" not in col #non contiene "id"
})

season = season[["season_year", "season_id", "championship_system_id"]]

teams = teams[["teams_name", "teams_nationality", "teams_country_code", "team_id"]]


with open("cleaned/Drivers_cleaned.csv", "w") as f:
    drivers.to_csv(f, index=False)

with open("cleaned/Teams_cleaned.csv", "w") as f:
    teams.to_csv(f, index=False)

with open("cleaned/Season_cleaned.csv", "w") as f:
    season.to_csv(f, index=False)


df = pd.merge(df, drivers, on="driver_id", how="left")
df = pd.merge(df, teams, on="team_id", how="left")
df = pd.merge(df, season, on="season_id", how="left")

df = endid(df)

with open("cleaned/Merged_drivers_info.csv", "w") as f:
    df.to_csv(f, index=False)

In [27]:
def checkforNAN(df):
    missing = df.isna() | df.astype("string").apply(
        lambda col: col.str.strip().eq("")
    )

    missing_cols = missing.columns[missing.any()].tolist()

    if missing_cols:
        print("columns with missing values:")
        for col in missing_cols:
            print(f"{col}: {missing[col].sum()} missing values")
    else:
        print("No missing values.")

In [33]:
#session round circuit

circuit = pd.read_csv("formula_one_circuit.csv").drop(
    columns=["api_id", "reference", "wikipedia"])
circuit = circuit.rename(columns={"id": "circuit_id"})
circuit = circuit[["name", "locality", "altitude", "longitude","country", "country_code", "circuit_id"]]
circuit = circuit.rename(columns={
    col: f"circuit_{col}" for col in circuit.columns
    if "id" not in col #non contiene "id"
}
)


round = pd.read_csv("formula_one_round.csv")
#cancellare le row che hanno la colonna "is_cancelled" = "t"
round = round[round["is_cancelled"] != "t"].drop(columns=["api_id", "season_id", "wikipedia", "is_cancelled"])


round = round.rename(columns={"id": "round_id"})
round = round.rename(columns={
    col: f"round_{col}" for col in round.columns
   if "id" not in col #non contiene "id"
})

round = round[["round_name", "round_number", "round_race_number", "round_date",  "circuit_id", "round_id"]]
#ordinare le righe da date più piccola a più grande
round = round.sort_values(by="round_date", ascending=True)

circuit = endid(circuit)
round = endid(round)

with open("cleaned/Circuit_cleaned.csv", "w") as f:
    circuit.to_csv(f, index=False)

with open("cleaned/Round_cleaned.csv", "w") as f:
    round.to_csv(f, index=False)










session_entry = pd.read_csv("formula_one_sessionentry.csv")

cols = ["id", "detail", "grid", "is_classified", "is_eligible_for_points", "laps_completed", "points", "position","session_id", "time"]
session_entry = session_entry[cols].dropna(subset=["grid"], how="all")   #droppare grid che sono vuote missing


td = pd.to_timedelta(session_entry["time"], errors="coerce")

# Tempo medio al giro per sessione
avg_lap_by_session = (
    td.div(session_entry["laps_completed"])
    .groupby(session_entry["session_id"])
    .transform("mean")
)

# Giri totali della sessione e giri mancanti per riga
max_laps = session_entry.groupby("session_id")["laps_completed"].transform("max")
missing_laps = max_laps - session_entry["laps_completed"]

# Tempo base della sessione (media dei tempi noti) + penalità per giri mancanti
base_session_time = td.groupby(session_entry["session_id"]).transform("mean")
estimated_td = base_session_time + (avg_lap_by_session * missing_laps)

filled_td = td.fillna(estimated_td)


def fmt_time(val):
    if pd.isna(val):
        return pd.NA
    total_seconds = val.total_seconds()
    hours = int(total_seconds // 3600)
    minutes = int((total_seconds % 3600) // 60)
    seconds = total_seconds % 60
    return f"{hours:02d}:{minutes:02d}:{seconds:04.1f}"


session_entry["time"] = filled_td.apply(fmt_time)

session_entry = session_entry.rename(columns={"id": "session_entry_id"})
session_entry = session_entry.rename(columns={
    col: f"session_entry_{col}" for col in session_entry.columns
    if "_id" not in col #non contiene "id"
})

#ordinare in modo sensato: session_entry_detail	grid	session_entry_is_classified	session_entry_is_eligible_for_points	session_entry_laps_completed	session_entry_points	session_entry_position	session_entry_time	session_entry_id	session_id
#time, position, grid ,points, lap,details, session_entry_is_classified,	session_entry_is_eligible_for_points
session_entry = session_entry[["session_entry_time", "session_entry_position", "session_entry_grid", "session_entry_points", "session_entry_laps_completed", "session_entry_detail", "session_entry_is_classified", "session_entry_is_eligible_for_points", "session_entry_id", "session_id"]]

session_entry = endid(session_entry)

with open("cleaned/Session_entry_cleaned.csv", "w") as f:
    session_entry.to_csv(f, index=False)



session = pd.read_csv("formula_one_session.csv")

cols = ["scheduled_laps", "type", "number", "round_id","id"]

mask = (
    (session["is_cancelled"] != "t")
    & (session["scheduled_laps"].notna())
    & (session["scheduled_laps"] != "NaN")
)


session = session.loc[mask, cols]
session = session.rename(columns={"id": "session_id"})
session = session.rename(columns={
    col: f"session_{col}" for col in session.columns
    if "id" not in col #non contiene "id"
})



driver_championship = pd.read_csv("formula_one_driverchampionship.csv")
cols = ["id", "driver_id", "highest_finish", "points", "position", "session_id","win_count","year"]

mask = (
    (driver_championship["is_eligible"] == "t")
    & (driver_championship["highest_finish"].notna())
    & (driver_championship["highest_finish"] != "NaN")
)

driver_championship = driver_championship.loc[mask, cols]

driver_championship = driver_championship.rename(columns={"id": "driver_championship_id"})
driver_championship = driver_championship.rename(columns={
    col: f"driver_championship_{col}" for col in driver_championship.columns
    if "id" not in col #non contiene "id"
})

#mergiare session id, poi così si aggiunge il round id e gli mergiamo i rounds e i circuits
driver_championship = pd.merge(driver_championship, session, on="session_id", how="left")
driver_championship = pd.merge(driver_championship, round, on="round_id", how="left")
driver_championship = pd.merge(driver_championship, circuit, on="circuit_id", how="left")

#droppare tutte le righe con circuit_id missing

driver_championship = driver_championship.dropna(subset=["circuit_id","driver_championship_position"])

driver_championship = endid(driver_championship)
#ordinare le righe in base alla data  dei round
driver_championship = driver_championship.sort_values(by="round_date", ascending=True)

with open("cleaned/Driver_Championship_cleaned.csv", "w") as f:
    driver_championship.to_csv(f, index=False)

#mergiare su session , session entry


with open("cleaned/Session_cleaned.csv", "w") as f:
    session.to_csv(f, index=False)

session = pd.merge(session, session_entry, left_on="session_id", right_on="session_id", how="left")
session = pd.merge(session, round, left_on="round_id", right_on="round_id", how="left")

#rimuovere le righe con session_entry_id missing, perché non hanno senso senza session_entry
session = session.dropna(subset=["session_entry_id"])



session = endid(session)
with open("cleaned/Merged_Session_info.csv", "w") as f:
    session.to_csv(f, index=False)



#unire a driver_championship, session_entry, e drivers in merged_Driver_chapionship_info 
#per entrambi i merge utilizzare un mapping come tra session e session_entry, per evitare di avere colonne duplicate con lo stesso nome

session_entry_clean = session_entry[["session_id", "session_entry_id", "session_entry_time", "session_entry_position", "session_entry_grid", "session_entry_points", "session_entry_laps_completed", "session_entry_detail", "session_entry_is_classified", "session_entry_is_eligible_for_points"]].dropna(subset=["session_id"]).drop_duplicates(subset=["session_id"], keep="first")

driver_championship = pd.merge(driver_championship, session_entry_clean, on="session_id", how="left")

# ora per i driver

driver_championship = pd.merge(driver_championship, drivers, left_on="driver_id", right_on="driver_id", how="left")

driver_championship = endid(driver_championship)

driver_championship = driver_championship[["drivers_forename", "drivers_surname", "drivers_date_of_birth", "drivers_nationality", "drivers_country_code", "driver_championship_highest_finish", "driver_championship_points", "driver_championship_position", "driver_championship_win_count", "driver_championship_year", "session_scheduled_laps", "session_type", "session_number", "session_entry_time", "session_entry_position", "session_entry_grid", "session_entry_points", "session_entry_laps_completed", "session_entry_detail", "session_entry_is_classified", "session_entry_is_eligible_for_points", "round_name", "round_number", "round_race_number", "round_date", "circuit_name", "circuit_locality", "circuit_altitude", "circuit_longitude", "circuit_country", "circuit_country_code", "driver_championship_id", "driver_id", "session_id", "round_id", "circuit_id", "session_entry_id"]]

with open("cleaned/Merged_Driver_Championship_info.csv", "w") as f:
    driver_championship.to_csv(f, index=False)


#session.head(10)
#averaging the not nan times, then multiplying by the number of completed laps?  

#checkforNAN(driver_championship)

#print(drivers.columns.tolist())